# Figure S5 — Host–virus Spearman correlations (viperin)

Standalone script for manuscript code availability. Computes pairwise host–virus
expression correlations in dinoflagellate scRNA-seq data, exports viperin–virus
significant pairs, and plots the Spearman correlation lollipop figure.

Outputs:

1. `host_virus_correlations.significant.viperin.tsv`
2. Lollipop plot PNG

No external project modules are required beyond standard Python packages.

In [9]:
from pathlib import Path
from typing import List, Optional, Sequence, Set, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import pearsonr, spearmanr

try:
    from statsmodels.stats.multitest import multipletests
except ImportError:
    multipletests = None

In [10]:
def read_emapper_annotations(path: Union[str, Path]) -> pd.DataFrame:
    """Load eggNOG-mapper annotation file (.emapper.annotations)."""
    header = None
    skiprows: List[int] = []
    with Path(path).open() as handle:
        for idx, line in enumerate(handle):
            if line.startswith("##"):
                skiprows.append(idx)
            elif line.startswith("#query"):
                header = line.lstrip("#").strip().split("\t")
                skiprows.append(idx)
                break
    if header is None:
        raise ValueError(f"No #query header line found in {path}")
    return pd.read_csv(
        path,
        sep="\t",
        names=header,
        skiprows=skiprows,
        dtype=str,
        low_memory=False,
    )


def read_viral_genome_eggnog(path: Union[str, Path]) -> pd.DataFrame:
    """Load tabular eggNOG output exported from viral genome ORFs."""
    header = None
    skiprows: List[int] = []
    with Path(path).open() as handle:
        for idx, line in enumerate(handle):
            if line.startswith("##"):
                skiprows.append(idx)
            elif line.startswith("#query"):
                header = line.lstrip("#").strip().split("\t")
                skiprows.append(idx)
                break
    if header is None:
        raise ValueError(f"No #query header line found in {path}")
    return pd.read_csv(
        path,
        sep="\t",
        names=header,
        skiprows=skiprows,
        dtype=str,
        low_memory=False,
    )


def _read_gene_list(path: Union[str, Path]) -> List[str]:
    with Path(path).open() as handle:
        return [line.strip() for line in handle if line.strip()]


def get_viral_genes(
    path_viral_list: Union[str, Path],
    eggnog: pd.DataFrame,
    path_ail3_list: Optional[Union[str, Path]] = None,
) -> Set[str]:
    """Union of viral gene IDs used for infection calling."""
    viral_genes = set(_read_gene_list(path_viral_list))
    if path_ail3_list:
        viral_genes.update(_read_gene_list(path_ail3_list))
    viral_subset = eggnog[eggnog["eggNOG_OGs"].str.contains("Viruses", na=False)]
    viral_genes.update(map(str, viral_subset["query"].unique()))
    return viral_genes


def define_infected(
    adata,
    viral_genes: Set[str],
    threshold: int = 10,
    counts_layer: str = "counts",
    status_key: str = "infection_status",
    viral_umi_key: str = "viral_UMIs",
):
    """Label cells as infected when total viral UMIs exceed `threshold`."""
    present = pd.Index(viral_genes).intersection(adata.var_names)
    if len(present) == 0:
        raise ValueError("None of the viral genes were found in adata.var_names.")
    x = adata.layers[counts_layer]
    viral_idx = adata.var_names.get_indexer(present)
    if sp.issparse(x):
        viral_umis = np.asarray(x[:, viral_idx].sum(axis=1)).ravel()
    else:
        viral_umis = np.asarray(x[:, viral_idx].sum(axis=1)).ravel()
    adata.obs[viral_umi_key] = viral_umis.astype(int)
    infected_mask = viral_umis >= int(threshold)
    adata.obs[status_key] = pd.Categorical(
        np.where(infected_mask, "infected", "uninfected"),
        categories=["uninfected", "infected"],
        ordered=True,
    )
    print(f"Infected cells: {int(infected_mask.sum())} / {adata.n_obs}")
    return adata


def _bh_adjust(pvalues: np.ndarray) -> np.ndarray:
    pvalues = np.asarray(pvalues, dtype=float)
    if multipletests is not None:
        _, adj, _, _ = multipletests(pvalues, method="fdr_bh")
        return adj
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    n = len(ranked)
    adj = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        adj[i] = prev
    out = np.empty(n, dtype=float)
    out[order] = np.clip(adj, 0, 1)
    return out


def _detection_fraction(counts_sub, gene_idx: int) -> float:
    column = counts_sub[:, gene_idx]
    if sp.issparse(column):
        expressed = column.getnnz(axis=0)
        expressed = int(expressed[0] if hasattr(expressed, "__len__") else expressed)
    else:
        expressed = int(np.count_nonzero(column))
    return expressed / counts_sub.shape[0] if counts_sub.shape[0] else 0.0


def _gene_layer_values(sub, gene: str, layer: str) -> np.ndarray:
    idx = sub.var_names.get_loc(str(gene))
    matrix = sub.layers[layer]
    column = matrix[:, idx]
    if sp.issparse(column):
        return np.asarray(column.todense()).ravel()
    return np.asarray(column).ravel()


def host_virus_correlation_pairs(
    adata,
    viral_genes: Set[str],
    cell_mask: Optional[pd.Series] = None,
    layer: str = "lognorm",
    counts_layer: str = "counts",
    min_host_detection: float = 0.05,
    min_virus_detection: float = 0.05,
) -> pd.DataFrame:
    """Pairwise host–virus correlations with Benjamini–Hochberg adjusted p-values."""
    if layer not in adata.layers or counts_layer not in adata.layers:
        raise KeyError("Expected counts and lognorm layers on adata.")

    viral_set = {str(g) for g in viral_genes}
    sub = adata[cell_mask].copy() if cell_mask is not None else adata.copy()
    n_cells = sub.n_obs
    if n_cells < 3:
        raise ValueError(f"Need at least 3 cells; found {n_cells}.")

    host_candidates = [
        str(g) for g in sub.var_names if str(g) not in viral_set
    ]
    virus_candidates = sorted(g for g in viral_set if g in sub.var_names)
    counts = sub.layers[counts_layer]

    host_ids, host_dets = [], []
    for gene in host_candidates:
        det = _detection_fraction(counts, sub.var_names.get_loc(gene))
        if det >= min_host_detection:
            host_ids.append(gene)
            host_dets.append(det)

    virus_ids, virus_dets = [], []
    for gene in virus_candidates:
        det = _detection_fraction(counts, sub.var_names.get_loc(gene))
        if det >= min_virus_detection:
            virus_ids.append(gene)
            virus_dets.append(det)

    host_det_map = dict(zip(host_ids, host_dets))
    virus_det_map = dict(zip(virus_ids, virus_dets))
    rows = []
    for host_id in host_ids:
        host_counts = _gene_layer_values(sub, host_id, counts_layer)
        host_expr = _gene_layer_values(sub, host_id, layer)
        if np.std(host_expr) == 0:
            continue
        for virus_id in virus_ids:
            virus_counts = _gene_layer_values(sub, virus_id, counts_layer)
            virus_expr = _gene_layer_values(sub, virus_id, layer)
            if np.std(virus_expr) == 0:
                continue
            n_double = int(((host_counts > 0) & (virus_counts > 0)).sum())
            pr, pp = pearsonr(host_expr, virus_expr)
            sr, sp_val = spearmanr(host_expr, virus_expr)
            rows.append(
                {
                    "host_id": host_id,
                    "virus_id": virus_id,
                    "n_cells": n_cells,
                    "n_double_positive": n_double,
                    "host_detection_frac": host_det_map[host_id],
                    "virus_detection_frac": virus_det_map[virus_id],
                    "pearson_r": float(pr),
                    "pearson_p": float(pp),
                    "spearman_r": float(sr),
                    "spearman_p": float(sp_val),
                }
            )

    pairs_df = pd.DataFrame(rows)
    if pairs_df.empty:
        return pairs_df
    pairs_df["pearson_p_adj"] = _bh_adjust(pairs_df["pearson_p"].to_numpy())
    pairs_df["spearman_p_adj"] = _bh_adjust(pairs_df["spearman_p"].to_numpy())
    pairs_df = pairs_df.sort_values("spearman_r", ascending=False).reset_index(drop=True)
    print(
        f"Host–virus correlation pairs: {len(pairs_df):,} "
        f"({len(host_ids)} host x {len(virus_ids)} viral; {n_cells} cells)"
    )
    return pairs_df

In [11]:
BLAST_COLUMNS = [
    "viral gene in genome", "query", "pident", "length",
    "mismatch", "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore",
]


def load_metag_genome_map(
    path: Union[str, Path],
    gene_ids: Optional[Sequence[str]] = None,
    min_pident: float = 90.0,
    min_length: int = 50,
) -> pd.DataFrame:
    """Load MetaG-to-viral-genome ORF mapping from AIL3.53_METAG.tsv."""
    hits = pd.read_csv(path, sep="\t", names=BLAST_COLUMNS)
    hits = hits[(hits["pident"] >= min_pident) & (hits["length"] >= min_length)]
    hits = hits.sort_values(["query", "bitscore"], ascending=[True, False])
    hits = hits.drop_duplicates("query", keep="first")
    if gene_ids is not None:
        hits = hits[hits["query"].isin(gene_ids)]
    return hits[["query", "viral gene in genome", "length"]].reset_index(drop=True)


def simplify_description(description: str) -> str:
    if description is None or pd.isna(description):
        return "Unknown function"
    text = str(description).strip()
    if text in {"#N/A", "-", "", "nan"}:
        return "Unknown function"
    if ". Source " in text:
        text = text.split(". Source ")[0]
    return text


def annotate_viperin_pairs(
    pairs: pd.DataFrame,
    metag_eggnog: pd.DataFrame,
    genome_eggnog: pd.DataFrame,
    genome_map: pd.DataFrame,
) -> pd.DataFrame:
    """Add manuscript annotation columns for the viperin correlation table."""
    out = pairs.copy()
    host_ann = metag_eggnog[["query", "Description"]].drop_duplicates("query")
    virus_ann = metag_eggnog[["query", "Description", "eggNOG_OGs", "max_annot_lvl"]].drop_duplicates("query")

    out = out.merge(
        host_ann,
        left_on="host_id",
        right_on="query",
        how="left",
    ).rename(columns={"Description": "host_description"}).drop(columns="query")

    out = out.merge(
        genome_map[["query", "viral gene in genome", "length"]],
        left_on="virus_id",
        right_on="query",
        how="left",
    ).drop(columns="query")

    genome_ann = genome_eggnog.rename(columns={"query": "viral gene in genome"})
    out = out.merge(
        genome_ann[["viral gene in genome", "eggNOG_OGs", "Description", "max_annot_lvl"]],
        on="viral gene in genome",
        how="left",
    )
    out = out.rename(columns={"eggNOG_OGs": "Eggnog annotation of viral gene"})

    fallback = out["virus_id"].map(
        virus_ann.set_index("query")["Description"].to_dict()
    )
    out["Description"] = out["Description"].fillna(fallback)
    out["Eggnog annotation of viral gene"] = out["Eggnog annotation of viral gene"].fillna(
        out["virus_id"].map(virus_ann.set_index("query")["eggNOG_OGs"].to_dict())
    )
    out["Description"] = out["Description"].fillna("#N/A")
    out["Eggnog annotation of viral gene"] = out["Eggnog annotation of viral gene"].fillna("#N/A")
    out["Simplified description"] = out["Description"].map(simplify_description)

    extra = out["length"].astype(str)
    missing = out["viral gene in genome"].isna()
    extra.loc[missing] = out.loc[missing, "virus_id"].map(
        virus_ann.set_index("query")["max_annot_lvl"].to_dict()
    )
    out[""] = extra

    column_order = [
        "host_id", "virus_id", "n_cells", "n_double_positive",
        "host_detection_frac", "virus_detection_frac",
        "pearson_r", "pearson_p", "spearman_r", "spearman_p",
        "pearson_p_adj", "spearman_p_adj",
        "viral gene in genome", "Eggnog annotation of viral gene",
        "Description", "Simplified description", "", "host_description",
    ]
    return out[column_order]

In [12]:
def _format_pvalue_label(p: float) -> str:
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def plot_host_virus_correlation_lollipop(
    pairs_table: pd.DataFrame,
    description_col: str = "Simplified description",
    save_path: Optional[Union[str, Path]] = None,
    show: bool = True,
    dpi: int = 300,
):
    """Horizontal lollipop plot of host–virus Spearman correlations."""
    df = pairs_table.sort_values("spearman_r", ascending=False).reset_index(drop=True)
    descriptions = df[description_col].fillna("").astype(str).str.strip()
    labels = [
        f"{vid} {desc.title()}" if desc else str(vid)
        for vid, desc in zip(df["virus_id"].astype(str), descriptions)
    ]
    x = df["spearman_r"].astype(float).to_numpy()
    p_adj = np.clip(df["spearman_p_adj"].astype(float).to_numpy(), 1e-300, 1.0)
    neg_log10_p = -np.log10(p_adj)
    n_coexpr = df["n_double_positive"].astype(float).to_numpy()

    legend_vals = (5, 10, 15)
    size_range = (40.0, 400.0)

    def _size_for_coexpr(n: float) -> float:
        frac = (n - legend_vals[0]) / (legend_vals[-1] - legend_vals[0])
        frac = float(np.clip(frac, 0.0, 1.0))
        return size_range[0] + frac * (size_range[1] - size_range[0])

    sizes = np.array([_size_for_coexpr(n) for n in n_coexpr])
    n_rows = len(df)
    fig, ax = plt.subplots(figsize=(12.5, max(4.0, 0.35 * n_rows + 1.5)), dpi=dpi)
    y_pos = np.arange(n_rows)
    cmap = plt.cm.YlGn
    norm = plt.Normalize(vmin=max(0.0, neg_log10_p.min()), vmax=max(1.0, neg_log10_p.max()))
    colors = cmap(norm(neg_log10_p))

    for yi, xi, size, color in zip(y_pos, x, sizes, colors):
        ax.hlines(yi, 0, xi, color="#888888", linewidth=1.2, zorder=1)
        ax.scatter(xi, yi, s=size, c=[color], edgecolors="0.25", linewidths=0.4, zorder=2)

    for yi, xi, pval in zip(y_pos, x, p_adj):
        label = _format_pvalue_label(float(pval))
        offset = 0.012 * (ax.get_xlim()[1] - ax.get_xlim()[0] or 1.0)
        ax.text(xi + offset, yi, label, va="center", ha="left", fontsize=9, color="#c0392b", clip_on=False)

    ax.axvline(0, color="0.35", linewidth=0.9, zorder=0)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Spearman correlation coefficient")
    ax.set_title("Host–virus Spearman correlations")
    ax.invert_yaxis()
    fig.tight_layout(rect=[0, 0, 0.66, 1])

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.04, fraction=0.025, shrink=0.62)
    cbar.set_label(r"$-\log_{10}$ (adj. $p$)")

    size_handles = [
        plt.scatter([], [], s=_size_for_coexpr(v) * 0.22, c="0.3", edgecolors="0.25", linewidths=0.4)
        for v in legend_vals
    ]
    cbar_pos = cbar.ax.get_position()
    fig.legend(
        size_handles,
        [str(v) for v in legend_vals],
        title="Co-expr. cells",
        loc="upper right",
        bbox_to_anchor=(cbar_pos.x1 + 0.061, cbar_pos.y0 - 0.01),
        bbox_transform=fig.transFigure,
        frameon=False,
        fontsize=8,
        title_fontsize=9,
    )

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved {save_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig, ax

In [13]:
# --- configuration ---
base_path = Path("/home/labs/vardi/amirz/pairing/Alps_All_MetaG")
output_dir = base_path / "gene_expression/dinoflagellata"
figure_dir = base_path / "gene_expression/figure_s5_correlation"

input_h5ad = base_path / "Alps_preprocessed_Dinoflagellata.h5ad"
output_viperin_tsv = output_dir / "host_virus_correlations.significant.viperin.tsv"
output_lollipop_png = figure_dir / "figure_s5_viperin_correlation_lollipop.png"

eggnog_file = (
    "/home/labs/vardi/Collaboration/databases/Gossenkollesee/"
    "MetaG_eggnog/MetaG_eggnog.emapper.annotations"
)
viral_gene_list = (
    "/home/labs/vardi/Collaboration/databases/Gossenkollesee/"
    "mapped_cells_virus/viral_genes.list"
)
path_ail3_list = base_path / "gene_expression/match_ail3_metag.list"
metag_genome_map_tsv = base_path / "gene_expression/AIL3.53_METAG.tsv"

viral_genome_eggnog = [
    base_path / "genomes_of_interest/proteins/AIL3.53.eggnog.tsv",
    base_path / "genomes_of_interest/proteins/AIL3.4.eggnog.tsv",
    base_path / "genomes_of_interest/proteins/AIL3.217.eggnog.tsv",
]

viperin_host = "140101_m_AIL3"
viral_umi_threshold = 10
min_host_detection = 0.05
min_virus_detection = 0.05
fdr_threshold = 0.05
min_abs_correlation = 0.2

### Load expression data and define infected cells

In [14]:
adata = sc.read_h5ad(input_h5ad)
eggnog = read_emapper_annotations(eggnog_file)
viral_genes = get_viral_genes(viral_gene_list, eggnog, path_ail3_list=path_ail3_list)
adata = define_infected(adata, viral_genes, threshold=viral_umi_threshold)
infected_mask = adata.obs["infection_status"] == "infected"
print(f"Analyzing {int(infected_mask.sum())} infected cells")
adata

Infected cells: 61 / 372
Analyzing 61 infected cells


AnnData object with n_obs × n_vars = 372 × 7248
    obs: 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_is_mito', 'log1p_total_counts_is_mito', 'pct_counts_is_mito', 'high_mito', 'viral_UMIs', 'infection_status'
    var: 'gene_ids', 'feature_types', 'n_counts', 'is_mito', 'is_ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mean', 'std'
    uns: 'log1p', 'neighbors', 'pca', 'tsne', 'umap'
    obsm: 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'lognorm'
    obsp: 'connectivities', 'distances'

### Compute host–virus correlation pairs

In [15]:
pairs_df = host_virus_correlation_pairs(
    adata,
    viral_genes=viral_genes,
    cell_mask=infected_mask,
    min_host_detection=min_host_detection,
    min_virus_detection=min_virus_detection,
)
pairs_df.head()

Host–virus correlation pairs: 52,624 (572 host x 92 viral; 61 cells)


,host_id,virus_id,n_cells,n_double_positive,host_detection_frac,virus_detection_frac,pearson_r,pearson_p,spearman_r,spearman_p,pearson_p_adj,spearman_p_adj
0,101929_g_TIL3J,48184_g_TPESL,61,45,0.737705,0.901639,0.891055,6.657959e-22,0.927228,7.581755e-27,1.698888e-17,3.989823e-22
1,39033_g_TPESL,9646_m_AIL3,61,27,0.540984,0.459016,0.889254,1.052022e-21,0.882311,5.716643e-21,1.698888e-17,1.504163e-16
2,39033_g_TPESL,42999_g_TPESL,61,31,0.540984,0.655738,0.853128,2.564156e-18,0.864058,3.077884e-19,2.248936e-14,5.399019e-15
3,44486_g_TPESL,36796_g_TPESL,61,26,0.475410,0.475410,0.854373,2.031910e-18,0.858778,8.761977e-19,2.138544e-14,1.152726e-14
4,48683_g_TPESL,9646_m_AIL3,61,27,0.524590,0.459016,0.850762,3.966911e-18,0.849991,4.566196e-18,2.982211e-14,4.805830e-14


### Filter viperin correlations and annotate for export

In [16]:
sig = pairs_df[
    (pairs_df["spearman_p_adj"] <= fdr_threshold)
    & (pairs_df["spearman_r"].abs() >= min_abs_correlation)
].copy()

viperin_cor = sig[sig["host_id"] == viperin_host].sort_values(
    "spearman_r", ascending=False
)
print(f"{len(viperin_cor)} viral genes correlate with viperin {viperin_host}")

genome_map = load_metag_genome_map(
    metag_genome_map_tsv,
    gene_ids=viperin_cor["virus_id"].tolist(),
)

genome_eggnog_df = pd.concat(
    [read_viral_genome_eggnog(path) for path in viral_genome_eggnog],
    ignore_index=True,
).drop_duplicates("query")

viperin_cor = annotate_viperin_pairs(viperin_cor, eggnog, genome_eggnog_df, genome_map)
viperin_cor

17 viral genes correlate with viperin 140101_m_AIL3


,host_id,virus_id,n_cells,n_double_positive,host_detection_frac,virus_detection_frac,pearson_r,pearson_p,spearman_r,spearman_p,pearson_p_adj,spearman_p_adj,viral gene in genome,Eggnog annotation of viral gene,Description,Simplified description,,host_description
0,140101_m_AIL3,193434_m_APESAF,61,17,0.278689,0.557377,0.620268,9.744055e-08,0.627290,6.316788e-08,2.018784e-05,0.000016,k127_4591788_1,"COG4581@1|root,KOG0949@2759|Eukaryota,1MD8K@12...",DEAD DEAH box RNA helicase. Source PGD,DEAD DEAH box RNA helicase,2277,Radical S-adenosyl methionine
1,140101_m_AIL3,42803_g_TPESL,61,11,0.278689,0.262295,0.680262,1.637346e-09,0.605326,2.367468e-07,7.049727e-07,0.000046,k127_1743399_5,"COG1078@1|root,KOG2681@2759|Eukaryota,38FGQ@33...",Deoxynucleoside triphosphate triphosphohydrola...,Deoxynucleoside triphosphate triphosphohydrola...,1341,Radical S-adenosyl methionine
2,140101_m_AIL3,40467_m_AIL3,61,14,0.278689,0.360656,0.588372,6.143983e-07,0.598993,3.402343e-07,9.211423e-05,0.000061,k127_3599416_3,"4QCQ4@10239|Viruses,4QVCT@35237|dsDNA viruses ...",Origin of replication binding protein,Origin of replication binding protein,636,Radical S-adenosyl methionine
3,140101_m_AIL3,56717_m_AIL3,61,8,0.278689,0.163934,0.545547,5.447764e-06,0.542354,6.334952e-06,5.471052e-04,0.000669,k127_6985368_2,"KOG4197@1|root,KOG4197@2759|Eukaryota,37QVI@33...",Pentatricopeptide repeat-containing protein,Pentatricopeptide repeat-containing protein,375,Radical S-adenosyl methionine
4,140101_m_AIL3,42998_g_TPESL,61,15,0.278689,0.557377,0.603169,2.680984e-07,0.537967,7.774409e-06,4.628908e-05,0.000793,k127_5984639_2,"4QB76@10239|Viruses,4QXGM@35237|dsDNA viruses ...",Ubiquitin-conjugating enzyme,Ubiquitin-conjugating enzyme,2124,Radical S-adenosyl methionine
5,140101_m_AIL3,6454_m_AIL3,61,14,0.278689,0.475410,0.574549,1.285516e-06,0.532521,9.984945e-06,1.654009e-04,0.000961,k127_1234125_1,"4QDVA@10239|Viruses,4QYSP@35237|dsDNA viruses ...",TBP-class protein binding,TBP-class protein binding,852,Radical S-adenosyl methionine
6,140101_m_AIL3,48182_g_TPESL,61,14,0.278689,0.426230,0.581668,8.826667e-07,0.531992,1.022836e-05,1.219218e-04,0.000980,k127_1246110_5,"4QB5I@10239|Viruses,4QUT2@35237|dsDNA viruses ...",N-methyltransferase activity,N-methyltransferase activity,1290,Radical S-adenosyl methionine
7,140101_m_AIL3,186805_m_AIL3,61,15,0.278689,0.590164,0.564847,2.116056e-06,0.530214,1.108686e-05,2.496756e-04,0.001053,k127_6043213_5,"4QFQ9@10239|Viruses,4QV8W@35237|dsDNA viruses ...",Pfam:RecQ_Zn_bind,Pfam:RecQ_Zn_bind,1464,Radical S-adenosyl methionine
8,140101_m_AIL3,165435_m_AIL3,61,17,0.278689,0.655738,0.485831,7.215354e-05,0.514931,2.176613e-05,4.710928e-03,0.001824,k127_431779_1,"4QEST@10239|Viruses,4QYC3@35237|dsDNA viruses ...",mRNA capping enzyme,mRNA capping enzyme,2238,Radical S-adenosyl methionine
9,140101_m_AIL3,72714_m_AIL3,61,11,0.278689,0.344262,0.518607,1.856143e-05,0.480350,8.935663e-05,1.519093e-03,0.005526,k127_1314342_3,"COG0270@1|root,COG0270@2|Bacteria,37ASC@32066|...",PFAM C-5 cytosine-specific DNA methylase,PFAM C-5 cytosine-specific DNA methylase,489,Radical S-adenosyl methionine


### Save table and lollipop plot

In [17]:
output_dir.mkdir(parents=True, exist_ok=True)
viperin_cor.to_csv(output_viperin_tsv, sep="\t", index=False)
print(f"Wrote {len(viperin_cor)} rows -> {output_viperin_tsv}")

plot_host_virus_correlation_lollipop(
    viperin_cor,
    description_col="Simplified description",
    save_path=output_lollipop_png,
    show=False,
)

Wrote 17 rows -> /home/labs/vardi/amirz/pairing/Alps_All_MetaG/gene_expression/dinoflagellata/host_virus_correlations.significant.viperin.tsv
Saved /home/labs/vardi/amirz/pairing/Alps_All_MetaG/gene_expression/figure_s5_correlation/figure_s5_viperin_correlation_lollipop.png


(<Figure size 3750x2235 with 2 Axes>,
 <Axes: title={'center': 'Host–virus Spearman correlations'}, xlabel='Spearman correlation coefficient'>)